# 🤖 Bulk Messaging Automation System

This notebook contains two automation systems built using Python:
1. **WhatsApp Web Bulk Messaging** — sends messages to contacts from an Excel file via WhatsApp Web using keyboard & mouse automation
2. **Gmail Bulk Email Sender** — sends emails to contacts from a CSV file via Gmail Web UI or Gmail SMTP

### 📦 Requirements
```
pip install pyautogui keyboard pandas openpyxl smtplib
```

### ⚠️ Important Notes
- Keep your screen active while automation is running
- Move mouse to **top-left corner** at any time to emergency stop (`pyautogui.FAILSAFE = True`)
- Press **I, I** on the cell or click **⬛** in VS Code toolbar to interrupt
- Gmail has a **500 emails/day** limit on free accounts
- WhatsApp Web must be open and logged in before running


## 📦 Section 1 — Imports
Importing all required libraries for automation, keyboard control, timing, and data handling.

In [2]:
import keyboard as k
import pyautogui
import time
import pandas as pd

## 📁 Section 2 — Working Directory
Setting and confirming the current working directory so file paths resolve correctly.

In [3]:
import os
print(os.chdir(os.getcwd()))

None


## 📊 Section 3 — Load WhatsApp Contacts
Loading contact numbers from an Excel file (`DemoCon_numbers.xlsx`).  
All values are read as strings to preserve leading zeros in phone numbers.


In [4]:
df = pd.read_excel("DemoCon_numbers.xlsx", dtype=str, header=None )

In [5]:
contact = df.iloc[:, 0].dropna().astype(str).tolist()

In [7]:
len(contact)

8

## ✍️ Section 4 — WhatsApp Message Templates
Define the message content to be sent to each contact.  
Text is split into multiple parts to simulate natural paragraph breaks using `shift+enter`.  
Supports **bold formatting** using WhatsApp's `*text*` syntax.


In [4]:
text1 = "Hello Sir,"

text2 = (
    "My name is *John Doe*, and I represent *ABC Solutions*. "
    "I hope you're doing well. I wanted to ask if you're currently looking to improve "
    "your *website, software, or business processes*."
)

text3 = (
    "We help businesses streamline their operations and improve efficiency through "
    "custom software, websites, and mobile applications."
)

text4 = (
    "Our team has worked with various organizations and businesses across different industries."
)

text5 = (
    "If this sounds relevant, I'd be happy to share a few ideas that may help your business. "
    "No obligation."
)

text6 = (
    "*Regards,*\n"
    "*John Doe*\n"
    "*Business Consultant*\n"
    "*ABC Solutions*"
)

## 🚀 Section 5 — WhatsApp Web Automation (Main Loop)
The main automation loop that:
1. Searches each contact in WhatsApp Web's search bar
2. Skips contacts not found (using image recognition on `not_found.png`)
3. Opens the chat and types the message in multiple parts
4. Sends the message and moves to the next contact

### 🖱️ Screen Coordinates Used
| Action | X | Y |
|--------|---|---|
| Search bar | 713 | 168 |
| First search result | 456 | 532 |
| Message input bar | 1064 | 1017 |

### ⏹️ To Stop
Move mouse to **top-left corner** of screen instantly stops the program.


In [10]:
time.sleep(10)
start = time.time()
i = 0
s = 0

for b in contact:

    pyautogui.click(713, 168)          # click the search bar
    time.sleep(0.4)
    pyautogui.hotkey('ctrl', 'a')      # select all existing text
    pyautogui.press('delete')          # clear it
    time.sleep(0.2)

    pyautogui.write(b, interval=0.05) 
    time.sleep(2.0)                   

    # ── Check if contact was NOT found ────────────────────────────
    try:
        not_found = pyautogui.locateOnScreen(
            "not_found.png",
            confidence=0.8
        )
        if not_found:
            # Clear search and skip this contact
            pyautogui.hotkey('ctrl', 'a')
            pyautogui.press('delete')
            pyautogui.press('escape')
            s += 1
            print(f"[SKIP] Not found: {b}")
            continue
    except pyautogui.ImageNotFoundException:
        pass
    except Exception as e:
        pass

    pyautogui.click(456, 532)          # click first contact in search results
    time.sleep(1.5)                  

    pyautogui.click(1064, 1017)        # click message input bar
    time.sleep(0.4)

    pyautogui.write(text1, interval=0.02)
    k.press_and_release('shift+enter')
    k.press_and_release('shift+enter')

    pyautogui.write(text2, interval=0.02)
    k.press_and_release('shift+enter')
    k.press_and_release('shift+enter')

    pyautogui.write(text3, interval=0.02)
    k.press_and_release('shift+enter')
    k.press_and_release('shift+enter')

    pyautogui.write(text4, interval=0.02)
    k.press_and_release('shift+enter')
    k.press_and_release('shift+enter')

    pyautogui.write(text5, interval=0.02)
    k.press_and_release('shift+enter')
    k.press_and_release('shift+enter')

    pyautogui.write(text6, interval=0.02)
    k.press_and_release('shift+enter')

    time.sleep(0.3)
    k.press_and_release('enter')       # SEND
    time.sleep(0.5)

    i += 1
    print(f"[SENT] {b}  ({i} total)")

    pyautogui.click(713, 168)
    time.sleep(0.3)
    pyautogui.hotkey('ctrl', 'a')
    pyautogui.press('delete')
    pyautogui.press('escape')
    time.sleep(0.5)

end = time.time()
print(f"\nDone in {end - start:.1f}s | Sent: {i} | Skipped: {s}")

[SENT] +923118036997  (1 total)
[SENT] +923335825437  (2 total)
[SENT] +923716726738  (3 total)
[SENT] +923003404342  (4 total)
[SENT] +923229179364  (5 total)
[SENT] +923009034444  (6 total)
[SENT] 9.2310954E+11  (7 total)
[SENT] 9.2348984E+11  (8 total)

Done in 163.1s | Sent: 8 | Skipped: 0


## 🖱️ Section 6 — Mouse Position Finder
Utility cell to find screen coordinates. Run it, then move your mouse to any element within 5 seconds to print its `(x, y)` position.

In [10]:
time.sleep(5)

cmp = pyautogui.position()
print(cmp.x)
print(cmp.y)

1185
558


## 📧 Section 10 — Gmail Bulk Sender via Web UI (BCC Method)
Sends the same email to all contacts using Gmail Web UI automation.  
All recipients are added to the **BCC field** in batches of 490 (Gmail's limit).

### ⚠️ Known Limitation
Gmail Web UI may only accept ~10 BCC entries reliably via automation.  
Use the **SMTP method** in Section 12 for large contact lists.


In [ ]:
import pyautogui
import time
import pandas as pd

csv_path = "test_contacts.csv"
contacts = pd.read_csv(csv_path)['email'].tolist()

# Split into chunks of 490 (safe limit under 500)
def chunk_list(lst, size=490):
    for i in range(0, len(lst), size):
        yield lst[i:i + size]

subject = "Testing Bulk Email automation"
your_email = "mohammadmohid1206@gmail.com"

text1 = "First paragraph of your email."
text2 = "Second paragraph."
text3 = "Third paragraph."

time.sleep(10)

start = time.time()
batch = 0

for chunk in chunk_list(contacts, 490):
    batch += 1
    print(f"\n[BATCH {batch}] Preparing {len(chunk)} recipients...")

    pyautogui.click(51,272)          # Compose button
    time.sleep(1.5)

    pyautogui.write(your_email, interval=0.05)
    time.sleep(0.3)
    pyautogui.press('tab')             # confirm your email as a chip
    time.sleep(0.4)

    pyautogui.click(1762,479)         # BCC button
    time.sleep(0.5)                    

    for email in chunk:
        pyautogui.write(email, interval=0.05)
        time.sleep(0.3)
        pyautogui.press('tab')         # confirm each email as a chip
        time.sleep(0.3)
        print(f"  [BCC] {email}")

    pyautogui.press('tab')             # BCC → Subject
    time.sleep(0.3)

    pyautogui.write(subject, interval=0.05)
    time.sleep(0.3)

    pyautogui.press('tab')             # Subject → Body
    time.sleep(0.3)

    pyautogui.write(text1, interval=0.02)
    pyautogui.hotkey('shift', 'enter')
    pyautogui.hotkey('shift', 'enter')

    pyautogui.write(text2, interval=0.02)
    pyautogui.hotkey('shift', 'enter')
    pyautogui.hotkey('shift', 'enter')

    pyautogui.write(text3, interval=0.02)
    pyautogui.hotkey('shift', 'enter')

    time.sleep(0.3)

    pyautogui.hotkey('ctrl', 'enter')
    time.sleep(2.0)                 

    print(f"[SENT] Batch {batch} — {len(chunk)} recipients")

end = time.time()
print(f"\nDone in {end - start:.1f}s | Total: {len(contacts)} | Batches: {batch}")

## 📧 Section 11 — Gmail Bulk Sender via Web UI (One Per Contact)
Sends one individual email per contact via Gmail Web UI.  
Each email is composed, addressed, and sent separately — no BCC involved.

### 🖱️ Coordinates Used
| Action | X | Y |
|--------|---|---|
| Compose button | 51 | 272 |
| Subject field click | 1188 | 527 |
| Subject confirm | 1185 | 558 |

### ⏹️ To Stop
Click **⬛** in VS Code toolbar or press **I, I** on the cell.  
Gmail rate limit: **1 email per 5 seconds** (enforced via `time.sleep(5.0)`).


In [6]:
import pyautogui
import time
import pandas as pd

csv_path = "test_contacts.csv"
contacts = pd.read_csv(csv_path)['email'].tolist()

subject = "Testing Bulk Email automation"
your_email = "testingmail846@gmail.com"

text1 = "First paragraph of your email."
text2 = "Second paragraph."
text3 = "Third paragraph."

print(f"Total contacts : {len(contacts)}")
print(f"⚠️  To stop: click ⬛ in VS Code or press I,I on the cell\n")

pyautogui.FAILSAFE = True      # move mouse to top-left corner = instant stop too
time.sleep(10)

start = time.time()
sent = 0
failed = 0

try:
    for email in contacts:

        pyautogui.click(51, 272)               # Compose button
        time.sleep(1.5)

        pyautogui.write(email, interval=0.05)
        # pyautogui.press('tab')                 # confirm email chip
        time.sleep(0.4)

        # pyautogui.press('tab')                 # To → CC  (skip)
        # time.sleep(0.2)
        # pyautogui.press('tab')
        pyautogui.click(1188, 527)
        pyautogui.click(1185, 558)                   # CC → Subject
        time.sleep(0.3)

        pyautogui.write(subject, interval=0.05)
        time.sleep(0.3)

        pyautogui.press('tab')                 # Subject → Body
        time.sleep(0.3)
        pyautogui.write(text1, interval=0.02)
        pyautogui.hotkey('shift', 'enter')
        pyautogui.hotkey('shift', 'enter')
        pyautogui.write(text2, interval=0.02)
        pyautogui.hotkey('shift', 'enter')
        pyautogui.hotkey('shift', 'enter')
        pyautogui.write(text3, interval=0.02)
        pyautogui.hotkey('shift', 'enter')
        time.sleep(0.3)

        pyautogui.hotkey('ctrl', 'enter')
        time.sleep(5.0)                        # Limit 1 mail per 5 seconds to avoid Gmail's sending limit

        sent += 1
        print(f"[SENT {sent}/{len(contacts)}] {email}")

except KeyboardInterrupt:
    print(f"\n🛑 Stopped manually")

finally:
    end = time.time()
    print(f"\nDone in {end - start:.1f}s | Sent: {sent} | Remaining: {len(contacts) - sent}")

Total contacts : 520
⚠️  To stop: click ⬛ in VS Code or press I,I on the cell

[SENT 1/520] test1@gmail.com
[SENT 2/520] test2@gmail.com
[SENT 3/520] test3@gmail.com
[SENT 4/520] test4@gmail.com
[SENT 5/520] test5@gmail.com
[SENT 6/520] info@codeclub.tech
[SENT 7/520] dummy.user001@gmail.com
[SENT 8/520] dummy.user002@gmail.com
[SENT 9/520] dummy.user003@gmail.com
[SENT 10/520] dummy.user004@gmail.com
[SENT 11/520] dummy.user005@gmail.com
[SENT 12/520] dummy.user006@gmail.com
[SENT 13/520] dummy.user007@gmail.com
[SENT 14/520] dummy.user008@gmail.com
[SENT 15/520] dummy.user009@gmail.com
[SENT 16/520] dummy.user010@gmail.com
[SENT 17/520] dummy.user011@gmail.com
[SENT 18/520] dummy.user012@gmail.com
[SENT 19/520] dummy.user013@gmail.com
[SENT 20/520] dummy.user014@gmail.com
[SENT 21/520] dummy.user015@gmail.com
[SENT 22/520] dummy.user016@gmail.com
[SENT 23/520] dummy.user017@gmail.com
[SENT 24/520] dummy.user018@gmail.com
[SENT 25/520] dummy.user019@gmail.com
[SENT 26/520] dummy.user0

## 📤 Section 12 — Gmail Bulk Sender via SMTP (Recommended)
Sends emails directly through Gmail's SMTP server — no browser needed.  
Faster, more reliable, and not affected by UI coordinate changes.

### 🔑 Setup — Gmail App Password
1. Go to **myaccount.google.com/apppasswords**
2. Enable **2-Step Verification** first if not already done
3. Create an App Password and paste it into `your_app_password`

### ✅ Advantages over Web UI
| Feature | Web UI | SMTP |
|---------|--------|------|
| Speed | Slow (UI typing) | Fast (direct server) |
| Reliability | Breaks if UI changes | Always stable |
| Daily limit | 500/day | 500/day |
| Needs browser open | ✅ Yes | ❌ No |


In [ ]:
import smtplib
import pandas as pd
import time
from email.mime.multipart import MIMEMultipart
from email.mime.text import MIMEText

csv_path        = "test_contacts.csv"
your_email      = "testingmail846@gmail.com"
your_app_password = "**** **** **** ****"  # Replace with your actual app password

subject = "Testing Bulk Email automation"

body = """First paragraph of your email.

Second paragraph.

Third paragraph.
"""

contacts = pd.read_csv(csv_path)['email'].tolist()
print(f"Loaded {len(contacts)} contacts")

start = time.time()
sent  = 0
failed = 0

with smtplib.SMTP_SSL('smtp.gmail.com', 465) as server:
    server.login(your_email, your_app_password)
    print("Logged in to Gmail SMTP\n")

    for email in contacts:
        try:
            # Build email
            msg = MIMEMultipart()
            msg['From']    = your_email
            msg['To']      = email
            msg['Subject'] = subject
            msg.attach(MIMEText(body, 'plain'))

            # Send
            server.sendmail(your_email, email, msg.as_string())
            sent += 1
            print(f"[SENT {sent}] {email}")

            time.sleep(5.0)  # Limit 1 mail per 5 seconds to avoid Gmail's sending limit

        except Exception as e:
            failed += 1
            print(f"[FAILED] {email} — {e}")

end = time.time()
print(f"\nDone in {end - start:.1f}s | Sent: {sent} | Failed: {failed}")

Loaded 520 contacts
Logged in to Gmail SMTP

[SENT 1] test1@gmail.com
[SENT 2] test2@gmail.com
[SENT 3] test3@gmail.com
[SENT 4] test4@gmail.com
[SENT 5] test5@gmail.com
[SENT 6] info@codeclub.tech
[SENT 7] dummy.user001@gmail.com
[SENT 8] dummy.user002@gmail.com
[SENT 9] dummy.user003@gmail.com
[SENT 10] dummy.user004@gmail.com
[SENT 11] dummy.user005@gmail.com
[SENT 12] dummy.user006@gmail.com
[SENT 13] dummy.user007@gmail.com
[SENT 14] dummy.user008@gmail.com
[SENT 15] dummy.user009@gmail.com
[SENT 16] dummy.user010@gmail.com
[SENT 17] dummy.user011@gmail.com
[SENT 18] dummy.user012@gmail.com
[SENT 19] dummy.user013@gmail.com
[SENT 20] dummy.user014@gmail.com
[SENT 21] dummy.user015@gmail.com
[SENT 22] dummy.user016@gmail.com
[SENT 23] dummy.user017@gmail.com
[SENT 24] dummy.user018@gmail.com
[SENT 25] dummy.user019@gmail.com
[SENT 26] dummy.user020@gmail.com
[SENT 27] dummy.user021@gmail.com
[SENT 28] dummy.user022@gmail.com
[SENT 29] dummy.user023@gmail.com
[SENT 30] dummy.user024

In [ ]:
import smtplib
import pandas as pd
import time
import os
from email.mime.multipart import MIMEMultipart
from email.mime.text import MIMEText
from email.mime.base import MIMEBase
from email import encoders

csv_path        = "testmail.csv"
your_email      = "testingmail846@gmail.com"
your_app_password = "**** **** **** ****"  # Replace with your actual app password

subject = "Testing Bulk Email automation"

body = """First paragraph of your email.

Second paragraph.
"""

attachment_paths = [
    "Mohammad_Mohid_Resume.pdf",
    "DemoCon_numbers.xlsx",
    "wallpaper.jpg"
]

contacts = pd.read_csv(csv_path)['email'].tolist()
print(f"Loaded {len(contacts)} contacts")

start = time.time()
sent = 0
failed = 0

with smtplib.SMTP_SSL('smtp.gmail.com', 465) as server:
    server.login(your_email, your_app_password)
    print("Logged in to Gmail SMTP\n")

    for email in contacts:
        try:
            msg = MIMEMultipart()
            msg['From']    = your_email
            msg['To']      = email
            msg['Subject'] = subject
            msg.attach(MIMEText(body, 'plain'))

            for path in attachment_paths:
                if os.path.exists(path):
                    with open(path, 'rb') as f:
                        part = MIMEBase('application', 'octet-stream')
                        part.set_payload(f.read())
                    encoders.encode_base64(part)
                    part.add_header(
                        'Content-Disposition',
                        f'attachment; filename={os.path.basename(path)}'
                    )
                    msg.attach(part)
                else:
                    print(f"⚠️  File not found, skipping: {path}")

            server.sendmail(your_email, email, msg.as_string())
            sent += 1
            print(f"[SENT {sent}] {email}")
            time.sleep(5.0)

        except Exception as e:
            failed += 1
            print(f"[FAILED] {email} — {e}")

end = time.time()
print(f"\nDone in {end - start:.1f}s | Sent: {sent} | Failed: {failed}")